# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [58]:
%pip install markdown

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [59]:
import os
import markdown as md_lib
from langchain_classic import hub
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langsmith import Client
import tiktoken
from IPython.display import HTML, display
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
encoding = tiktoken.encoding_for_model('gpt-4o-mini')

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.invoke({'query': query})
    if isinstance(result, list):
        result_text = ' '.join([str(item) for item in result])
    elif isinstance(result, dict):
        result_text = result.get('content', '') or str(result)
    else:
        result_text = str(result)
    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:2000])
    return trimmed

# LangChain tool
tools = [
    Tool(
        name='TavilySafeSearch',
        func=safe_search,
        description='Web search tool that trims output to avoid token overflow with GPT-4o-mini'
    )
]

# Pull the ReAct prompt from LangSmith hub
client = Client()
prompt = client.pull_prompt('hwchase17/react', dangerously_pull_public_prompt=True)

# Agent executor
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True,
                               handle_parsing_errors=True, max_iterations=1)

# Lightweight helper: one Tavily search + one direct LLM call
def run_task(user_prompt: str) -> str:
    first_line = next(line.strip() for line in user_prompt.strip().splitlines() if line.strip())
    search_results = safe_search(first_line)
    messages = [
        SystemMessage(content=(
            "You are a helpful research assistant. "
            "Use the provided search results to answer the user's request. "
            "Format your response in clean Markdown."
        )),
        HumanMessage(content=f"{user_prompt}\n\nSearch results:\n{search_results}")
    ]
    return llm.invoke(messages).content

# Display helper: renders Markdown with a white background
def show(md_text: str):
    html_body = md_lib.markdown(md_text, extensions=['tables', 'fenced_code'])
    display(HTML(f"""
    <div style="background-color:#ffffff;color:#1a1a1a;padding:1.5em 2em;
                border-radius:8px;line-height:1.7;border:1px solid #e0e0e0;
                font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
        {html_body}
    </div>
    """))


### Note: `run_task` vs `AgentExecutor`

In this lab we use a custom helper called `run_task` instead of calling `agent_executor` directly for each exercise. Here is why.

**The problem with `AgentExecutor` under a restricted API tier**

The standard ReAct agent loop works by letting the LLM decide how many search rounds it needs. Each round appends the previous thought, action, and observation to a growing *scratchpad* that is sent back to the LLM on every call. With three rounds this scratchpad can exceed **10 000 tokens** — the per-minute token limit (TPM) for a free or low-tier OpenAI organisation. When that limit is hit, the 429 error is silently caught by `handle_parsing_errors=True`, the iteration counter increments without doing any useful work, and the executor eventually returns `"Agent stopped due to iteration limit or time limit"` instead of a real answer.

**What `run_task` does differently**

`run_task` replaces the multi-step loop with a fixed two-step pipeline:

1. **One Tavily search** — calls `safe_search` directly and trims the result to 2 000 tokens.
2. **One LLM call** — passes the user prompt together with the search results to `ChatOpenAI` and returns the formatted answer.

This keeps every request well under 3 000 tokens in total, which is safe on any API tier. The trade-off is that the model cannot refine its search with follow-up queries, but for the structured summaries required in this lab a single well-targeted search is sufficient.

`agent_executor` is still defined in the setup cell to show the full agent-building pattern taught in the lesson; it is simply not invoked in the exercises here due to the token budget constraint.


### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [60]:
# Your prompt here
prompt_1 = """
Search the web for the most recent advancements in generative AI applied to healthcare in 2025.
Then provide a structured Markdown summary that includes:

## Latest Generative AI Advancements in Healthcare (2025)

- At least 5 key developments or breakthroughs
- For each development: what it is, which organization is behind it, and its potential patient impact
- A brief **Trends & Outlook** section at the end

Use real-time search results. Format the output in clean Markdown with headers, bullet points, and bold highlights.
"""

# Run it
response_1 = run_task(prompt_1)


In [61]:
show(response_1)


### Exercise 2: AI Startups Landscape

Goal: Track 2025's top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [62]:
# Your prompt here
prompt_2 = """
Search the web for the top emerging AI startups of 2025 and their most notable innovations.
Return a structured Markdown report:

## Top Emerging AI Startups of 2025

For each startup include:
- **Company name** and founding year (if available)
- **Core product or innovation**
- **Key differentiator** -- what makes them stand out
- **Funding or valuation** (if known)
- Source URL

List at least 6 companies. Format everything in clean, readable Markdown.
"""

# Run it
response_2 = run_task(prompt_2)


In [63]:
show(response_2)


### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI's ChatGPT Team and Anthropic's Claude Pro.
- Ensure the agent outputs a structured response in Markdown.

In [64]:
# Your prompt here
prompt_3 = """
Search the web for the latest information on OpenAI's ChatGPT Team plan and Anthropic's Claude Pro subscription.
Then produce a structured Markdown comparison:

## ChatGPT Team vs Claude Pro: Feature & Pricing Comparison (2025)

Include clearly organized sections covering:
- **Pricing** (monthly/annual cost per user)
- **Key features** unique to each plan
- **Model access** (which AI models are included)
- **Context window and usage limits**
- **Collaboration and team features**
- **User reviews and sentiment** from recent sources
- **Overall verdict**: which plan is better for which type of user

Format everything in clean, readable Markdown with a summary table if possible.
"""

# Run it
response_3 = run_task(prompt_3)


In [65]:
show(response_3)


Feature/Plan,ChatGPT Team,Claude Pro
Pricing,$100/month per user,$20/month (or $17/month annually)
Key Features,"Enhanced Codex usage, team collaboration tools","Access to Opus, Sonnet, Haiku, unlimited projects, cross-conversation memory"
Model Access,"Codex, GPT-4","Claude 2, Claude Code"
Context Window,Not specified,Not specified
Usage Limits,5x more Codex usage than Plus,5x more usage than Pro
Collaboration Features,Team collaboration tools,"Cowork feature, unlimited projects"
User Reviews & Sentiment,"Generally positive, focused on productivity","Positive, praised for coding assistance"
Overall Verdict,Best for teams needing high Codex usage,Ideal for individual professionals with moderate usage


## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.

**My Use Case: Global AI Regulation Tracker**

Keeping up with fast-moving AI laws and policies worldwide is challenging for developers and companies. An agent that searches and summarizes the latest regulatory developments by region gives practitioners a quick, structured briefing they would otherwise need hours to compile manually.

In [66]:
# Bonus: Global AI Regulation Tracker
prompt_bonus = """
Search the web for the latest AI regulation and policy developments worldwide in 2025,
including major laws passed, proposed regulations, and government responses to AI risks.

Provide a structured Markdown briefing:

## Global AI Regulation Tracker: 2025 Update

Organize by region (EU, US, Asia, Rest of World). For each entry include:
- **Legislation or policy name**
- **What it regulates** (e.g., high-risk AI, deepfakes, foundation models)
- **Current status** (draft / enacted / under review)
- **Key impact** on AI companies and developers

End with a **Trends & Takeaways** section summarizing the overall direction of global AI governance.
Format everything in clean, structured Markdown.
"""

response_bonus = run_task(prompt_bonus)


In [67]:
show(response_bonus)
